# Demo tóm tắt văn bản tiếng Việt

## 1. Kết nối Drive

In [ ]:
from pathlib import Path
from google.colab import drive

demo_run_id = ""

drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")

if not drive_root.is_dir():
    raise FileNotFoundError("Lỗi: Không tìm thấy thư mục dự án trên Drive.")


## 2. Lấy dữ liệu mới nhất thông qua GitHub

In [ ]:
import os
import sys
import subprocess

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = ""

if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)

remote = subprocess.run(
    ["git", "remote", "get-url", "origin"], cwd=repo_dir,
    capture_output=True, text=True, check=True,
)
if remote.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository không đúng dự án.")

status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=repo_dir,
    capture_output=True, text=True, check=True,
)
if status.stdout.strip() != "":
    raise RuntimeError("Lỗi: Repository còn thay đổi. Hãy dùng runtime mới.")

subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)
if target_commit == "":
    result = subprocess.run(
        ["git", "rev-parse", "origin/main"], cwd=repo_dir,
        capture_output=True, text=True, check=True,
    )
    target_commit = result.stdout.strip()
subprocess.run(["git", "checkout", "--detach", target_commit], cwd=repo_dir, check=True)
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
print("Commit demo:", target_commit)


## 3. Cài thư viện

In [ ]:
%pip uninstall -y -q diffusers torchvision timm fastai
%pip install -q "jedi==0.20.1"
%pip install -q -r /content/vietnamese-summarization/requirements.txt
%pip check


## 4. Chọn model và đọc bản tham chiếu

In [ ]:
import json
from app.demo_data import select_completed_run, load_validation_examples
from src.utils import load_train_config, load_generation_config

profile, training_summary, checkpoint_run_dir = select_completed_run(drive_root, demo_run_id)
demo_examples = load_validation_examples(drive_root, profile)

runtime_dir = Path("/content/vietnamese_summarization_demo") / profile["run_id"]
runtime_dir.mkdir(parents=True, exist_ok=True)
demo_config_path = runtime_dir / "train.json"
demo_generation_path = runtime_dir / "generation.json"

demo_config_path.write_text(
    json.dumps(profile["train_config"], ensure_ascii=False, indent=2), encoding="utf-8",
)
demo_generation_path.write_text(
    json.dumps(profile["generation_config"], ensure_ascii=False, indent=2), encoding="utf-8",
)
demo_config = load_train_config(demo_config_path)
demo_generation = load_generation_config(demo_generation_path)

print("Run demo:", profile["run_id"])
print("Best epoch:", training_summary["best_epoch"])
print("Best validation loss:", training_summary["best_validation_loss"])
print("Số mẫu validation có bản tham chiếu:", len(demo_examples))
print("Giới hạn nguồn/đích:", demo_config["max_source_length"], demo_config["max_target_length"])
print("Generation:", demo_generation)


## 5. Tải best lên GPU

In [ ]:
import torch
from app.gradio_app import DemoSummarizer, create_demo

if not torch.cuda.is_available():
    raise RuntimeError("Lỗi: Cần kết nối runtime GPU để tải model demo.")

if "demo_summarizer" not in globals():
    demo_summarizer = DemoSummarizer(
        checkpoint_run_dir=checkpoint_run_dir,
        config_path=demo_config_path,
        generation_config_path=demo_generation_path,
    )
    loaded_demo_run_id = profile["run_id"]
elif loaded_demo_run_id != profile["run_id"]:
    raise RuntimeError("Lỗi: Đã chọn run khác. Hãy dùng phiên mới để tải model.")

print("Model:", loaded_demo_run_id)
print("Thiết bị:", next(demo_summarizer.model.parameters()).device)


## 6. Mở giao diện

In [ ]:
if "demo" in globals():
    demo.close()

demo = create_demo(
    summarizer=demo_summarizer,
    examples=demo_examples,
    run_id=loaded_demo_run_id,
)
demo.launch(share=True, inline=True)
